In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

print(fake.head())
print(real.head())

In [ ]:
fake["label"] = 0
real["label"] = 1

In [ ]:
df = pd.concat([fake, real])

print(df.shape)

In [ ]:
df = df[["text","label"]]

In [ ]:
pip install nltk

In [ ]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    return text

In [ ]:
df["text"] = df["text"].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(df["text"])

y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

accuracy = accuracy_score(y_test,pred)

print("Accuracy:",accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,pred)

print(cm)

In [ ]:
import pickle

pickle.dump(model,open("model.pkl","wb"))

pickle.dump(
    vectorizer,
    open("vectorizer.pkl","wb")
)

In [ ]:
news = """
Scientists discovered a magic pill
that makes humans immortal.
"""

cleaned = clean_text(news)

vector = vectorizer.transform([cleaned])

prediction = model.predict(vector)

print(prediction)

In [ ]:
news = """
Scientists discovered a magic pill
that makes humans immortal.
"""

cleaned = clean_text(news)

vector = vectorizer.transform([cleaned])

prediction = model.predict(vector)[0]

if prediction == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

In [ ]:
pip install flask

In [ ]:
from flask import Flask,render_template,request
import pickle

app = Flask(__name__)

model = pickle.load(open("model.pkl","rb"))
vectorizer = pickle.load(open("vectorizer.pkl","rb"))

In [ ]:
@app.route("/",methods=["GET","POST"])

def home():

    result=""

    if request.method=="POST":

        news=request.form["news"]

        vec=vectorizer.transform([news])

        pred=model.predict(vec)

        result="Real News" if pred[0]==1 else "Fake News"

    return render_template(
        "index.html",
        result=result
    )

In [ ]:
pip install shap

In [ ]:
pip install streamlit